# 6.1 Simple RNN network in PyTorch

In [8]:
import torch
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1) # Output layer

    def forward(self, x, h0):
        out, hn = self.rnn(x, h0)
        out = self.fc(out[:, -1, :]) # Use the last time step's output
        return out, hn

# Hyperparameters
input_size = 10
hidden_size = 20
num_layers = 1
sequence_length = 5
batch_size = 3

# Create the model
model = SimpleRNN(input_size, hidden_size, num_layers)

# Example input sequence (batch_size, sequence_length, input_size)
input_seq = torch.randn(batch_size, sequence_length, input_size)

# Initial hidden state (num_layers, batch_size, hidden_size)
h0 = torch.zeros(num_layers, batch_size, hidden_size)

# Forward pass through the RNN
output, hn = model(input_seq, h0)
print("Input shape:", input_seq.shape)
print("Output shape:", output.shape)
print("Hidden state shape:", hn.shape)

# Example of using the model for a simple prediction task
x = torch.randn(1, sequence_length, input_size)  # Single sample
h0 = torch.zeros(num_layers, 1, hidden_size)
prediction, _ = model(x, h0)
print("Prediction:", prediction.item())

Input shape: torch.Size([3, 5, 10])
Output shape: torch.Size([3, 1])
Hidden state shape: torch.Size([1, 3, 20])
Prediction: -0.019854899495840073


# 6.2 LSTM in PyTorch

In [9]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out, (hn, cn)

# Hyperparameters
input_size = 10
hidden_size = 20
num_layers = 2
output_size = 1
sequence_length = 5
batch_size = 3

# Create model instance
model = LSTMModel(input_size, hidden_size, num_layers, output_size)

# Example input sequence
input_seq = torch.randn(batch_size, sequence_length, input_size)

# Forward pass
output, (hn, cn) = model(input_seq)

# Print shapes
print("Input shape:", input_seq.shape)
print("Output shape:", output.shape)
print("Hidden state shape:", hn.shape)
print("Cell state shape:", cn.shape)

# Example of using the model for a simple prediction task
x = torch.randn(1, sequence_length, input_size)  # Single sample
prediction, _ = model(x)
print("Prediction:", prediction.item())

Input shape: torch.Size([3, 5, 10])
Output shape: torch.Size([3, 1])
Hidden state shape: torch.Size([2, 3, 20])
Cell state shape: torch.Size([2, 3, 20])
Prediction: 0.20730344951152802


# 6.3 GRU in PyTorch

In [10]:
import torch
import torch.nn as nn

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

# Hyperparameters
input_size = 10
hidden_size = 20
num_layers = 2
output_size = 1
sequence_length = 5
batch_size = 3

# Create model instance
model = GRUModel(input_size, hidden_size, num_layers, output_size)
# Example input sequence
input_seq = torch.randn(batch_size, sequence_length, input_size)
# Forward pass
output = model(input_seq)
# Print shapes
print("Input shape:", input_seq.shape)
print("Output shape:", output.shape)
# Example of using the model for a simple prediction task
x = torch.randn(1, sequence_length, input_size)  # Single sample
prediction = model(x)
print("Prediction:", prediction.item())

Input shape: torch.Size([3, 5, 10])
Output shape: torch.Size([3, 1])
Prediction: 0.044928453862667084


# 6.4 RNN in TensorFlow

In [13]:
import tensorflow as tf
import numpy as np

# Define hyperparameters
batch_size = 32
sequence_length = 10
input_size = 8
hidden_units = 16
output_size = 4

# Create synthetic input data
input_data = tf.random.normal([batch_size, sequence_length, input_size])

# Define an RNN layer
rnn_layer = tf.keras.layers.SimpleRNN(units=hidden_units, return_sequences=True, return_state=True)

# Define a model using the Functional API
inputs = tf.keras.Input(shape=(sequence_length, input_size))
rnn_output, final_state = rnn_layer(inputs)
outputs = tf.keras.layers.Dense(output_size)(rnn_output)
model = tf.keras.Model(inputs=inputs, outputs=[outputs, final_state])

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Generate synthetic target data
target_output = np.random.randn(batch_size, sequence_length, output_size)
target_final_state = np.random.randn(batch_size, hidden_units)

# Train the model
history = model.fit(
    input_data,
    [target_output, target_final_state],
    epochs=5,
    batch_size=batch_size
)

# Make predictions
predictions, final_state_pred = model.predict(input_data)

# Print shapes and sample outputs
print("Input Shape:", input_data.shape)
print("RNN Output Shape:", predictions.shape)
print("RNN Final State Shape:", final_state_pred.shape)
print("Sample Prediction (first sequence, first timestep):")
print(predictions[0, 0])
print("Sample Final State:")
print(final_state_pred[0])

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 543ms/step - dense_2_loss: 1.8219 - loss: 3.4444 - simple_rnn_2_loss: 1.6225
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - dense_2_loss: 1.8094 - loss: 3.4286 - simple_rnn_2_loss: 1.6192
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - dense_2_loss: 1.7971 - loss: 3.4130 - simple_rnn_2_loss: 1.6159
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - dense_2_loss: 1.7849 - loss: 3.3976 - simple_rnn_2_loss: 1.6126
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - dense_2_loss: 1.7729 - loss: 3.3823 - simple_rnn_2_loss: 1.6093
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Input Shape: (32, 10, 8)
RNN Output Shape: (32, 10, 4)
RNN Final State Shape: (32, 16)
Sample Prediction (first sequence, first timestep):
[ 0.5138832  -1.0057961  -0.29562974  0.37939382]
Sample Final State:
[-0.6221144   0.46190652 -0.814191    0.9301006  -0.9431395   0.9304459
 -0.94938976  0.3488002  -0.39225787 -0.24078102 -0.6152821  -0.31932858
 -0.90833306  0.4281782  -0.3712296  